In [1]:
from scone_tools.evaluation.reconstruction_evaluation import best_permutation_similarity
from simulate_data import *
from scone_tools.algorithms.SCoNE import SCoNE_parallel


In [53]:
import tqdm
for n_features in [5,10,15,20,30,50]:
    similarity_lst = []
    for i in tqdm.tqdm(range(10)):
        views = simulate_views(n=1000, M_G=n_features, M_C=n_features, 
                           rank=3, seed=i, gamma=5, noise=0.5,
                          )
    
        factor_matrices, loss_function, benchmark_info = SCoNE_parallel(
            views['G'].astype(float), 
            views['C'].astype(float), 
            views['Z'].astype(float), 
            rank=3,
            alpha=max(views['G'].max(),views['C'].max())**2,
            lambda_H_G=1e-4, 
            lambda_H_C=1e-4, 
            lambda_Gloss=1,  
            num_init=3,
            init='nndsvda',
            G_loss_type='kl_div', 
            C_loss_type='kl_div', 
            #use_gpu=True
        )
        similarity, permutation = best_permutation_similarity(views['W'],factor_matrices['W'])
        similarity_lst.append(similarity)
    print(f'n_features {n_features}, similarity {np.mean(similarity_lst)}, se {np.std(similarity_lst, ddof=1)/np.sqrt(len(similarity_lst))}')


100%|██████████| 10/10 [02:08<00:00, 12.87s/it]


n_features 5, similarity 0.6370063966712023, se 0.025714759271364778


100%|██████████| 10/10 [03:37<00:00, 21.77s/it]


n_features 10, similarity 0.7587973105173182, se 0.013376295070061862


100%|██████████| 10/10 [04:58<00:00, 29.86s/it]


...

n_features 20, similarity 0.8671880172271994, se 0.007456248927087072


100%|██████████| 10/10 [10:04<00:00, 60.50s/it]


n_features 30, similarity 0.9207603421066862, se 0.005558827318544324


  0%|          | 0/10 [01:24<?, ?it/s]


KeyboardInterrupt: 

In [2]:
matrices = simulate_views(n=1000, M_G=20, M_C=20, rank=3, seed=0, noise=0.5)
views = simulate_views(n=1000, M_G=20, M_C=20, rank=3, seed=0, noise=0.5)

In [3]:
for k,v in matrices.items():
    assert np.allclose(v, views[k])

In [56]:
v

array([[0., 0., 0., ..., 6., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 3., 1., 0.],
       ...,
       [0., 1., 3., ..., 7., 0., 0.],
       [0., 0., 0., ..., 1., 1., 0.],
       [0., 1., 0., ..., 1., 0., 1.]], shape=(1000, 20))

In [57]:
views[k]

array([[ 0.,  0.,  0., ...,  2.,  0.,  0.],
       [ 0.,  1.,  1., ...,  1.,  0.,  0.],
       [ 3.,  2.,  3., ...,  4.,  0.,  0.],
       ...,
       [ 0.,  0.,  9., ..., 14.,  3.,  1.],
       [ 0.,  0.,  1., ...,  0.,  1.,  0.],
       [ 4.,  0.,  0., ...,  1.,  0.,  1.]], shape=(1000, 20))

why does noise matter so much

In [4]:
import numpy as np

def proj_nonneg(x):
    # If already feasible, return x as-is (avoids an allocation most iterations)
    if x.min() >= 0:
        return x
    return np.maximum(x, 0)

def make_correlated_matrices(r, X, y_nonneg=False, seed=0):
    # random matrix always created from N(0,1) distribution
    rng = np.random.default_rng(seed)
    Y = rng.normal(size=X.shape)
    if y_nonneg:
        Y = proj_nonneg(Y)
    return_mat = r*X + np.sqrt(1 - r**2) * Y
    if r==1:
        assert np.allclose(X, return_mat)
    return return_mat

In [7]:
n=1000
M_G=20
M_C=20
rank=3
seed=0
gamma=5
noise=0.1
sparsity=0.2
rG=0
rGC=0
rZ=0
signed_cov_effects=False
subgroup_structure=True

overdispersion_nu=0

rng = np.random.default_rng(seed)

if subgroup_structure:
    # 1. Simulate W by random assignment to rank subgroups
    sizes = np.full(rank, n//rank) # evenly balance subgroups
    sizes[:n % rank] += 1
    labels = np.repeat(np.arange(rank), sizes)
    rng.shuffle(labels)
    W = np.zeros((n, rank), dtype=int)
    W[np.arange(n), labels] = 1
    
    # 2. Simulate H_G with genetic architecture similarity rG
    Sigma = (1-rG)*np.eye(rank,rank) + np.full((rank,rank),rG)
    H_G = proj_nonneg(rng.multivariate_normal(np.zeros((rank)), Sigma, size=M_G))
    
    # 3. Simulate H_C with similarity to genetic subgroups rGC
    H_C = proj_nonneg(make_correlated_matrices(rGC, H_G, seed=seed))
    
    # 4. impose sparsity (P(W_ij=0)=sparsity)
    mask = np.random.rand(*(M_C,rank)) > sparsity
    H_C = H_C * mask
    mask = np.random.rand(*(M_G,rank)) > sparsity
    H_G = H_G * mask

    # 5. Generate covariate with similarity to subgroup 1 structure
    z = make_correlated_matrices(rZ, W[:,1], y_nonneg=True, seed=seed)

else: 
    assert rZ==0, "set rZ to 0 if no subgroup structure"
    z = proj_nonneg(rng.normal(size=n))

# 5. Generate covariate matrix
assert (z>=0).all()
Z = np.column_stack([np.ones(len(z)), z])

# 6. Simulate covariate effects
U_C = rng.normal(size=(M_C, Z.shape[1]))
U_G = rng.normal(size=(M_G, Z.shape[1]))
if not signed_cov_effects:
    U_C = proj_nonneg(U_C)
    U_G = proj_nonneg(U_G)

# 7. Generate matrix means
if subgroup_structure:
    mu_C = W@H_C.T + Z@U_C.T + (noise)*proj_nonneg(rng.normal(size=(n, M_C)))
    # standardize A and B to have same marginal standard deviation
    A = W @ H_G.T
    B = Z @ U_G.T
    mu_G = A + gamma*(np.std(A) / np.std(B))*B + (noise)*proj_nonneg(rng.normal(size=(n, M_G)))
else:
    mu_C = Z@U_C.T + (noise)*proj_nonneg(rng.normal(size=(n, M_C)))
    mu_G = Z@U_G.T + (noise)*proj_nonneg(rng.normal(size=(n, M_G)))

In [9]:
np.std(A)

np.float64(0.45223174659682147)

In [10]:
np.std(B)

np.float64(0.4656637969672694)

In [11]:
np.std((noise)*proj_nonneg(rng.normal(size=(n, M_G))))

np.float64(0.05800947836976276)

In [12]:
np.std(W@H_C.T)

np.float64(0.5351739752442007)

In [13]:
np.std(Z@U_C.T)

np.float64(0.8367512422916695)

In [14]:
np.std((noise)*proj_nonneg(rng.normal(size=(n, M_C))))

np.float64(0.05828863172083485)

In [15]:
U_C

array([[0.        , 0.        ],
       [0.        , 0.75438572],
       [0.        , 0.46814891],
       [0.52675577, 1.37544531],
       [0.        , 1.73860211],
       [1.26881527, 0.57306599],
       [2.38359223, 0.2049786 ],
       [0.82147892, 0.        ],
       [1.13435741, 0.16782597],
       [0.        , 2.1169392 ],
       [0.        , 0.00886133],
       [0.        , 0.        ],
       [0.5312724 , 0.7384096 ],
       [0.35437253, 0.        ],
       [1.007997  , 0.        ],
       [0.        , 0.60328096],
       [0.56284942, 0.        ],
       [2.47243568, 0.        ],
       [0.        , 0.        ],
       [1.42079984, 0.        ]])

In [16]:
U_G

array([[0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.70795552],
       [1.02045589, 0.        ],
       [0.23922797, 0.78820724],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.10186287, 0.        ],
       [0.63112909, 0.        ],
       [0.41262736, 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.42659546, 0.74659486],
       [0.15808271, 1.71349089],
       [0.        , 0.52059642],
       [0.        , 0.2346295 ],
       [0.        , 1.11512338],
       [0.17602091, 1.1922445 ],
       [0.        , 0.        ],
       [0.        , 0.        ]])